In [1]:
import string
import numpy as np
import pandas as pd
import networkx as nx
import rustworkx as rx
import torch
import copy
from qiskit import QuantumCircuit, execute
from qiskit.compiler import transpile
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit import DAGInNode, DAGOpNode, DAGOutNode
from qiskit.providers.aer import AerSimulator
from qiskit.providers.fake_provider import FakeMontreal, FakeWashington, FakeSherbrooke, FakeToronto, FakeCambridgeAlternativeBasis
from qiskit.transpiler.passes import RemoveBarriers
from qiskit.transpiler.passes import RemoveFinalMeasurements
from torch_geometric.utils.convert import from_networkx

import pickle 
from pathlib import Path

In [2]:
from data_preparation import circ_dag_converter_v2 as circ_dag_converter
from helper import get_openqasm_gates
GATE_DICT = {item: index for index, item in enumerate(get_openqasm_gates())}
# NUM_ERROR_DATA = 4
NUM_NODE_TYPE = 2 + len(GATE_DICT)

In [6]:
backends = [('FakeAlmaden', 20), ('FakeBoeblingen', 20), ('FakeBrooklyn', 65), ('FakeCairo', 27), ('FakeCambridge', 28), ('FakeCambridgeAlternativeBasis', 28), ('FakeGuadalupe', 16), ('FakeHanoi', 27), ('FakeJohannesburg', 20), ('FakeKolkata', 27), ('FakeManhattan', 65), ('FakeMontreal', 27), ('FakeMumbai', 27), ('FakeParis', 27), ('FakeRochester', 53), ('FakeSingapore', 20), ('FakeSydney', 27), ('FakeToronto', 27), ('FakeWashington', 127)]
backends_dict = dict(backends)

In [7]:
import importlib, json
from functools import lru_cache
from qiskit_aer.noise import NoiseModel

_FAKE_MODS = ["qiskit.providers.fake_provider"]

@lru_cache(None)
def get_backend(device_name: str):
    for modname in _FAKE_MODS:
        try:
            mod = importlib.import_module(modname)
            cls = getattr(mod, device_name)
            return cls()
        except (ModuleNotFoundError, AttributeError):
            continue
    raise ValueError(f"Unknown fake backend: {device_name}")


In [8]:
def get_training_data(dirpath, backend_name, n_qubit, pattern="*.qasm", key2id=None, cut=None):
    G_list = []
    GATE_DICT = {item: index for index, item in enumerate(get_openqasm_gates())}
    NUM_NODE_TYPE = 2 + len(GATE_DICT)
    
    backend = get_backend(backend_name)
    for p in sorted(Path(dirpath).glob(pattern)):
        # print(str(p).split('/')[-1].split('.')[0])
        cir_id = str(p).split('/')[-1].split('.')[0]
        if cut and int(cir_id.split('_')[0]) != key2id[cut]:
            continue
        # print(cut, cir_id)
        try:
            qc = QuantumCircuit.from_qasm_file(p)
            transpiled_qc = transpile(
                qc,
                backend=backend, 
                seed_transpiler=1997
            )
            # print(transpiled_qc)
            G = circ_dag_converter.circ_to_dag_with_data(transpiled_qc, backend_name, n_qubit)
            G_new = circ_dag_converter.networkx_torch_convert(G, NUM_NODE_TYPE + 14)
            edge_attr = circ_dag_converter.get_edge_features_matrix_calib(
                G,
                n_qubit,
                include_node_redundant=False
            )
            G_new.edge_attr = edge_attr
            G_new.cir_id = cir_id
            # print(G_new)
            G_list.append(G_new)
        except Exception as e:
            print(e)

    if cut:
        state_csv_path = dirpath + f"{backend_name}_{cut}.csv"
        g_list_path = dirpath + f"G_list_{cut}.pkl"
        table_by_cirid_path = dirpath + f"table_by_cirid_{cut}.pkl"
    else:
        state_csv_path = dirpath + f"{backend_name}.csv"
        g_list_path = dirpath + f"G_list.pkl"
        table_by_cirid_path = dirpath + f"table_by_cirid.pkl"

    df = pd.read_csv(state_csv_path, dtype={'circuit': 'string'}).dropna()
    df["circuit"] = df["circuit"].astype(str)
    
    table_by_cirid = {
        cid: sub[["POS","POF","Target Value"]].to_dict("records")
        for cid, sub in df.groupby("circuit")
    }

    idx_to_remove = []
    for i, G in enumerate(G_list):
        if G.cir_id not in table_by_cirid:
            print(G.cir_id)
            idx_to_remove.append(i)

    G_list[:] = [G for i, G in enumerate(G_list) if i not in idx_to_remove]

    print(state_csv_path)
    print(g_list_path)
    print(table_by_cirid_path)

    with open(g_list_path, 'wb') as f:
        pickle.dump(G_list, f)
    
    with open(table_by_cirid_path, 'wb') as f:
        pickle.dump(table_by_cirid, f)
        
    return G_list, table_by_cirid

# pretrain

In [1]:
for backend_name in backends_dict:
    dirpath = f"./larger_circuits/data/baseline_training_data/{backend_name}/"
    G_list, table_by_cirid = get_training_data(dirpath, backend_name, backends_dict[backend_name])
    

In [2]:
G_list

# finetune

In [1]:
from pathlib import Path
GATE_DICT = {item: index for index, item in enumerate(get_openqasm_gates())}
NUM_NODE_TYPE = 2 + len(GATE_DICT)

pattern = "*.qasm"
key2id = {'cnot':0, 'addition':1, 'permutation':2, 'phase':3,
 'ghz':4, 'simon':5, 'qft':6, 'similarity':7, 'expression':8}

for backend_name in backends_dict:
    for cut in ['addition', 'phase', 'ghz', 'simon', 'qft', 'similarity']:
        dirpath = f"./larger_circuits/data/baseline_tunning_data/{backend_name}/"
        G_list, table_by_cirid = get_training_data(dirpath, backend_name, n_qubit=backends_dict[backend_name], pattern="*.qasm", key2id=key2id, cut=cut)

        # print(len(G_list))
        # for G in G_list:
        #    print(G)

        # print(table_by_cirid)
    
    # break

# test-RQ1

In [2]:
from pathlib import Path
GATE_DICT = {item: index for index, item in enumerate(get_openqasm_gates())}
NUM_NODE_TYPE = 2 + len(GATE_DICT)

pattern = "*.qasm"
key2id = {'cnot':0, 'addition':1, 'permutation':2, 'phase':3,
 'ghz':4, 'simon':5, 'qft':6, 'similarity':7, 'expression':8}

for backend_name in backends_dict:
    for cut in ['addition', 'phase', 'ghz', 'simon', 'qft', 'similarity']:
        dirpath = f"./larger_circuits/data/evaluation_data/{backend_name}/"
        G_list, table_by_cirid = get_training_data(dirpath, backend_name, n_qubit=backends_dict[backend_name], pattern="*.qasm", key2id=key2id, cut=cut)

    #     print(len(G_list))
    #     for G in G_list:
    #        print(G)

    #     print(table_by_cirid)
    
    # break

# test-RQ3

In [33]:
def get_test_data(dirpath, backend_name, n_qubit, pattern="*.qasm", key2id=None, cut=None):
    G_list = []
    GATE_DICT = {item: index for index, item in enumerate(get_openqasm_gates())}
    NUM_NODE_TYPE = 2 + len(GATE_DICT)
    
    backend = get_backend(backend_name)
    for p in sorted(Path(dirpath).glob(pattern)):
        # print(str(p).split('/')[-1].split('.')[0])
        cir_id = str(p).split('/')[-1].split('.')[0]
        if cut and int(cir_id.split('_')[0]) != key2id[cut]:
            continue
        # print(cut, cir_id)
        try:
            qc = QuantumCircuit.from_qasm_file(p)
            transpiled_qc = transpile(
                qc,
                backend=backend, 
                seed_transpiler=1997
            )
            # print(transpiled_qc)
            G = circ_dag_converter.circ_to_dag_with_data(transpiled_qc, backend_name, n_qubit)
            G_new = circ_dag_converter.networkx_torch_convert(G, NUM_NODE_TYPE + 14)
            edge_attr = circ_dag_converter.get_edge_features_matrix_calib(
                G,
                n_qubit,
                include_node_redundant=False
            )
            G_new.edge_attr = edge_attr
            G_new.cir_id = cir_id
            # print(G_new)
            G_list.append(G_new)
        except Exception as e:
            print(e)

    if cut:
        state_csv_path = dirpath + f"{backend_name}_{cut}.csv"
        g_list_path = dirpath + f"G_list_{cut}.pkl"
        table_by_cirid_path = dirpath + f"table_by_cirid_{cut}.pkl"
    else:
        state_csv_path = dirpath + f"{backend_name}.csv"
        g_list_path = dirpath + f"G_list.pkl"
        table_by_cirid_path = dirpath + f"table_by_cirid.pkl"

    with open(table_by_cirid_path, 'rb') as f:
        table_by_cirid = pickle.load(f)

    idx_to_remove = []
    for i, G in enumerate(G_list):
        if G.cir_id not in table_by_cirid:
            print(G.cir_id)
            idx_to_remove.append(i)

    G_list[:] = [G for i, G in enumerate(G_list) if i not in idx_to_remove]

    print(state_csv_path)
    print(g_list_path)
    print(table_by_cirid_path)

    with open(g_list_path, 'wb') as f:
        pickle.dump(G_list, f)
    
    
        
    return G_list, table_by_cirid

In [7]:
from pathlib import Path
GATE_DICT = {item: index for index, item in enumerate(get_openqasm_gates())}
NUM_NODE_TYPE = 2 + len(GATE_DICT)

pattern = "*.qasm"
key2id = {'cnot':0, 'addition':1, 'permutation':2, 'phase':3,
 'ghz':4, 'simon':5, 'qft':6, 'similarity':7, 'expression':8}

for backend_name in backends_dict:
    for cut in ['addition', 'phase', 'ghz', 'simon', 'qft', 'similarity']:
        dirpath = f"./larger_circuits/data/testing_data/{backend_name}/"
        G_list, table_by_cirid = get_test_data(dirpath, backend_name, n_qubit=backends_dict[backend_name], pattern="*.qasm", key2id=key2id, cut=cut)

    #     print(len(G_list))
    #     for G in G_list:
    #        print(G)

    #     print(table_by_cirid)
    
    # break

# test summary

In [38]:
def get_test_result(dir, seed, backends_dict, cut_list):
    result = {}
    for backend in backends_dict:
        hld = []
        for cut in cut_list:
            all = torch.load(f"{dir}/seed_{seed}/{backend}/{cut}/all.pth", weights_only=False)
            for k, v in all['test_hld_per_circuit'].items():
                hld.append(v)
        result[backend] = np.mean(hld)

    return result
    

In [3]:
dir =  './larger_circuits/model/evaluation_data'
cut_list = ['addition', 'phase', 'ghz', 'simon', 'qft', 'similarity']

for i in range(1, 4):
    print(get_test_result(dir, i, backends_dict, cut_list))


In [16]:
def get_test_result_by_cut(dir, seed, backends_dict, cut_list):
    result = {}
    for cut in cut_list:
        hld = []
        for backend in backends_dict:
            all = torch.load(f"{dir}/seed_{seed}/{backend}/{cut}/all.pth", weights_only=False)
            for k, v in all['test_hld_per_circuit'].items():
                hld.append(v)
        result[cut] = np.mean(hld)

    return result

In [4]:
dir =  './larger_circuits/model/evaluation_data'
cut_list = ['addition', 'phase', 'ghz', 'simon', 'qft', 'similarity']

result = get_test_result_by_cut(dir, 3, backends_dict, cut_list)
result

In [5]:
all_path = './larger_circuits/model/evaluation_data/seed_3/full_model'
for cut in ['addition', 'phase', 'ghz', 'simon', 'qft', 'similarity']:
    all = torch.load(f"{all_path}/{cut}/all.pth", weights_only=False)
    result[cut] = {}
    for k, v in all['test_hld_per_circuit'].items():
        result[cut][k] = v

# print(np.mean(hld))

# for k, v in result.items():
#     print(k, np.mean(v))
final_result = {}
for key in backends_dict:
    final_result[key] = []
for key, value in result.items():
    for k in value:
        final_result[k.split('_')[0]].append(value[k])

for key, value in final_result.items():
    print(key, np.mean(value))


In [6]:
for key, value in result.items():
    temp = []
    for l, v in value.items():
        temp.append(v)
    print(key, np.mean(temp))

# QOIN

In [59]:
import numpy as np
import pandas as pd


def to_scalar(x):
    arr = np.asarray(x).reshape(-1)
    if arr.size == 0:
        return np.nan
    return float(arr[0])


def summarize_by_backend(results):
    rows = []

    for backend, bench_dict in results.items():
        all_values = []

        for values in bench_dict.values():
            flat_values = [to_scalar(v) for v in values]
            flat_values = [v for v in flat_values if not np.isnan(v)]
            all_values.extend(flat_values)

        count = len(all_values)
        mean_val = float(np.mean(all_values)) if count > 0 else np.nan
        std_val = float(np.std(all_values, ddof=1)) if count > 1 else 0.0

        rows.append({
            "backend": backend,
            "mean": mean_val,
            # "std": std_val
        })

    return pd.DataFrame(rows).sort_values("backend").reset_index(drop=True)


def summarize_by_benchmark(results):
    benchmark_map = {}

    for bench_dict in results.values():
        for benchmark, values in bench_dict.items():
            if benchmark not in benchmark_map:
                benchmark_map[benchmark] = []

            flat_values = [to_scalar(v) for v in values]
            flat_values = [v for v in flat_values if not np.isnan(v)]
            benchmark_map[benchmark].extend(flat_values)

    rows = []
    for benchmark, all_values in benchmark_map.items():
        count = len(all_values)
        mean_val = float(np.mean(all_values)) if count > 0 else np.nan
        std_val = float(np.std(all_values, ddof=1)) if count > 1 else 0.0

        rows.append({
            "benchmark": benchmark,
            "mean": mean_val,
            # "std": std_val
        })

    return pd.DataFrame(rows).sort_values("benchmark").reset_index(drop=True)




In [74]:
with open('./larger_circuits/results/seed_5/saveRQ1.json', 'rb') as f:
    qoin_result = pickle.load(f)